In [8]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# from tensorflow.keras import mixed_precision

# print("TensorFlow Version:", tf.__version__)
# print("GPU Available:", tf.config.list_physical_devices('GPU'))

# mixed_precision.set_global_policy('mixed_float16')
# gpus = tf.config.list_physical_devices('GPU')
# if gpus:
#     try:
#         for gpu in gpus:
#             tf.config.experimental.set_memory_growth(gpu, True)
#         print("GPU Memory Growth Enabled")
#     except RuntimeError as e:
#         print(e)

In [ ]:
from pathlib import Path

# DEST_DIR = Path('/content/drive/MyDrive/NULP/FF++/299')
MODEL_NAME = "DeepFakeDetection"
DEST_DIR = Path(f'/content/drive/MyDrive/NULP/FF++/{MODEL_NAME}')

print("Setting up Data Generators...")

datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    # validation_split=0.2,
    # brightness_range=[0.8, 1.2] 
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    # brightness_range=[0.8, 1.2]
)

DEST_TRAIN = DEST_DIR / "training_balanced"
DEST_VAL = DEST_DIR / "validation_balanced"
DETS_TEST = DEST_DIR / "test_balanced"
DEST_EVAL = DEST_DIR / "eval_balanced"

SIZE = (299, 299)

train_generator = datagen.flow_from_directory(
    DEST_TRAIN, target_size=SIZE, batch_size=32, class_mode='binary', color_mode="rgb",
)
val_generator = datagen.flow_from_directory(
    DEST_VAL, target_size=SIZE, batch_size=32, class_mode='binary', shuffle=False, color_mode="rgb",
)
test_generator = datagen.flow_from_directory(
    DETS_TEST, target_size=SIZE, batch_size=32, class_mode='binary', shuffle=False, color_mode="rgb",
)

eval_generator = datagen.flow_from_directory(
    DEST_EVAL, target_size=SIZE, batch_size=32, class_mode='binary', shuffle=False, color_mode="rgb",
)

# train_generator = datagen.flow_from_directory(
#     dest_dir, target_size=SIZE, batch_size=32, class_mode='binary', subset='training'
# )
# val_generator = datagen.flow_from_directory(
#     dest_dir, target_size=SIZE, batch_size=32, class_mode='binary', subset='validation'
# )
print(" - Generators ok")

In [ ]:
labels = train_generator.classes
counts = np.bincount(labels)
total = len(labels)

class_weight = {
    0: total / (2.0 * counts[0]),
    1: total / (2.0 * counts[1]),
}

print(f"Class weight count -> {counts[0]}, 1 -> {counts[1]}")
print("class_weight:", class_weight)

In [ ]:
from keras.applications import EfficientNetB0
from keras.layers import Input

print("Building EfficientNetB0...")

base_model = EfficientNetB0(
    weights="imagenet", include_top=False, input_shape=(SIZE[0], SIZE[1], 3)
)
base_model.trainable = False

inputs = Input(shape=(SIZE[0], SIZE[1], 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
predictions = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=predictions)

# === Head ===
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

model.fit(train_generator, epochs=5, validation_data=val_generator, class_weight=class_weight)

# === Fine-tuning ===
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.TruePositives(name="tp"),
        tf.keras.metrics.FalsePositives(name="fp"),
        tf.keras.metrics.TrueNegatives(name="tn"),
        tf.keras.metrics.FalseNegatives(name="fn"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(f"./models/efficientNetB0_{MODEL_NAME}.keras", save_best_only=True, monitor="val_loss"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

history = model.fit(
    train_generator, epochs=30, validation_data=val_generator, callbacks=callbacks, class_weight=class_weight
)

In [ ]:
np.save(f"./models/efficientNetB0_{MODEL_NAME}.npy", history)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy', color='blue', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='green', linewidth=2)
plt.title('Fine-Tuned XceptionNet Accuracy', fontsize=14)
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss', color='blue', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', color='green', linewidth=2)
plt.title('Fine-Tuned XceptionNet Loss', fontsize=14)
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['auc'], label='Training AUC', color='blue', linewidth=2)
plt.plot(history.history['val_auc'], label='Validation AUC', color='green', linewidth=2)
plt.title('Fine-Tuned XceptionNet AUC', fontsize=14)
plt.ylabel('AUC')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
def print_metrics(y_true, y_probs, threshold=0.5 ,tf = "============== In"):
    y_pred = (y_probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    print(tf)

    print(f"\n" + "="*45)
    print(f"Metrics (Threshold = {threshold:.4f})")
    print("="*45)
    print(f"• AUC-ROC:       {auc_score:.4f}")
    print(f"• Accuracy:      {accuracy * 100:.2f}%")
    print(f"• Precision:     {precision:.4f}")
    print(f"• Recall:        {recall:.4f}")
    print(f"• F1-Score:      {f1:.4f}")
    print("-" * 45)
    print(f"Confusion Matrix:")
    print(f"  [TN: {tn:4d}  |  FP: {fp:4d}]")
    print(f"  [FN: {fn:4d}  |  TP: {tp:4d}]")
    print("="*45)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve


y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())
print(f"Classes found: {test_generator.class_indices}")

print("\nMaking prediction..")
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred_probs = y_pred_probs.ravel()

auc_score = roc_auc_score(y_true, y_pred_probs)

In [ ]:
print("in")
print_metrics(y_true, y_pred_probs, threshold=0.5)

fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

if abs(optimal_threshold - 0.5) > 0.05:
    print(f"\nBetter threshold based on ROC-curve: {optimal_threshold:.4f}")
    print_metrics(y_true, y_pred_probs, threshold=optimal_threshold)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

y_true = eval_generator.classes
class_names = list(eval_generator.class_indices.keys())
print(f"Classes found: {eval_generator.class_indices}")

print("\nMaking prediction...")
y_pred_probs = model.predict(eval_generator, verbose=1)
y_pred_probs = y_pred_probs.ravel()

auc_score = roc_auc_score(y_true, y_pred_probs)

In [ ]:
print("out")
print_metrics(y_true, y_pred_probs, threshold=0.5, tf = "============== Out")

fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

if abs(optimal_threshold - 0.5) > 0.05:
    print(f"\nBetter threshold based on ROC-curve: {optimal_threshold:.4f}")
    print_metrics(y_true, y_pred_probs, threshold=optimal_threshold)